# Per-LD-block GwasSumStats construction

## Description

For one study's GWAS TSV, fan out per LD block, build a `pecotmr::GwasSumStats` (including `summaryStatsQc()`), and serialize one RDS per `(study, block)`. The RDS files become the per-block GWAS inputs for `twas.ipynb` and `ctwas.ipynb`.

## Inputs

- `--study` &mdash; study identifier.
- `--gwas-tsv` &mdash; path to the GWAS summary-statistics TSV.
- `--ld-blocks` &mdash; BED file of LD-block intervals (`chr/chrom`, `start`, `stop/end`).
- `--ld-meta` &mdash; LD-meta TSV pointing at per-region LD references.
- `--genome` &mdash; genome build label. Default `hg19`.
- `--modular-script-dir` &mdash; directory containing the worker R scripts.

## Output

- `{cwd}/{study}.{block_id}.gwas_sumstats.rds` per LD block (`block_id` sanitises `:`→`_`, `-`→`_`).


## Example

```bash
sos run pipeline/gwas_sumstats.ipynb gwas_sumstats_construct \
    --cwd output --modular-script-dir /path/to/code/script \
    --study TEST_GWAS \
    --gwas-tsv input/twas/protocol_example.twas.gwas_sumstats.chr22.tsv.gz \
    --ld-blocks input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --ld-meta input/ld_reference/protocol_example.ld_meta_file.tsv
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: gwas_tsv = path
parameter: ld_blocks = path
parameter: ld_meta = path
parameter: genome = 'hg19'
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1

In [ ]:
[generate_manifest]
# Parse the LD-blocks BED into a (region, region_id) TSV manifest via
# ld_blocks_to_manifest.R. Downstream fans out over its rows; no Python
# parsing in this notebook.
input: ld_blocks
output: f"{cwd}/{study}.blocks_manifest.tsv"
task: trunk_workers = 1, trunk_size = 1, walltime = '15m', mem = '2G', cores = 1, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/ld_blocks_to_manifest.R \
        --ld-blocks ${_input} \
        --output ${_output}

In [ ]:
[gwas_sumstats_construct]
# Fan out over the manifest's rows: one GwasSumStats per LD block.
import csv
jobs = list(csv.DictReader(open(f"{cwd}/{study}.blocks_manifest.tsv"), delimiter='\t'))
input: gwas_tsv, for_each = 'jobs'
output: f"{cwd}/{study}.{_jobs['region_id']}.gwas_sumstats.rds"
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/gwas_sumstats_construct.R \
        --study ${study} \
        --gwas-tsv ${_input} \
        --ld-block ${_jobs['region']} \
        --ld-meta ${ld_meta} \
        --genome ${genome} \
        --output ${_output}